# Reddit Experiment Data Processing

Extracts data from the political signaling username reddit experiment and puts it into an inference-ready format.

In [1]:
from pathlib import Path
from pyprojroot import here

import json
import polars as pl

In [2]:
# Needed to filter out non-experimental posts
experiment_subreddits = [
    "Baking",
    "bodyweightfitness",
    "explainlikeimfive",
    "NoStupidQuestions",
    "AskDocs", 
    "cooking", 
    "TravelHacks", 
    "netflix", 
    "AskCulinary",
    "LearnProgramming", 
    "backpacking", 
    "camping", 
    "socialskills",
    "booksuggestions", 
    "Fantasy", 
    "Cycling", 
    "Sleep", 
    "CasualConversation",
    "AskMen", 
    "hiking", 
    "Houseplants", 
    "LanguageLearning", 
    "Photography",
    "frugal",
    "selfimprovement",
    "Pets",
    "Skiing", 
    "IWantToLearn",
    "Beauty", 
    "GiftIdeas"
]

In [11]:
experiment_results = Path(here("data/experiment_results"))
reply_schema = {
    'id': pl.String,
    'subreddit': pl.String,
    'username': pl.String,
    'username_score': pl.Int64,
    'label': pl.String,
    'content': pl.String
    }
replies = pl.DataFrame(schema=reply_schema)

for user in experiment_results.iterdir():
    if not user.is_dir():
        continue

    username = user.name
    label = "liberal" if "no_kings" in username else "conversative" if "MAGA" in username else "neutral"
    username_score = -1 if label == "liberal" else 1 if label == "conversative" else 0

    for post in user.iterdir():
        if not post.is_dir():
            continue

        subreddit, id = [s[::-1] for s in post.name[::-1].split('_', maxsplit=1)]
        collect = post / "24h.json"
        if not collect.is_file():
            continue

        with collect.open("r") as f:
            collect_dict = json.load(f)
            contents = [repr(comments["body"]) for comments in collect_dict["comments"]]
            if not contents:
                continue

            starter_dict = {
                "id": id,
                "subreddit": subreddit,
                "username": username,
                "username_score": username_score,
                "label": label
            }
            new_rows = pl.DataFrame([starter_dict] * len(contents))
            new_rows = new_rows.with_columns(pl.Series(contents).alias("content"))
            replies = pl.concat([replies, new_rows])

# Move label column to the end
replies = replies.select([pl.exclude("label"), "label"])

replies


id,subreddit,username,username_score,content,label
str,str,str,i64,str,str
"""AskMen""","""1rvbenb""","""Fresh_Window_6484""",0,"""'Your submission was removed b…","""neutral"""
"""Pets""","""1rsbi9c""","""Fresh_Window_6484""",0,"""'If you shut her out she will …","""neutral"""
"""Pets""","""1rsbi9c""","""Fresh_Window_6484""",0,"""'Why is she waking you? Is for…","""neutral"""
"""CasualConversation""","""1s048db""","""Fresh_Window_6484""",0,"""'Whilst most lightning is clou…","""neutral"""
"""CasualConversation""","""1s048db""","""Fresh_Window_6484""",0,"""'I havent been able to stop te…","""neutral"""
…,…,…,…,…,…
"""cycling""","""1rpon7j""","""Evening_Issue_8448""",0,"""'Just be aware of the general …","""neutral"""
"""cycling""","""1rpon7j""","""Evening_Issue_8448""",0,"""""I like old mtbs so I always c…","""neutral"""
"""cycling""","""1rpon7j""","""Evening_Issue_8448""",0,"""'Check for frame cracks, fork …","""neutral"""


In [13]:
replies.write_parquet(here("data/experiment.parquet"))